In [1]:
import os
import warnings
import torch
import numpy as np
import tensorflow as tf
from PIL import Image
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TFBertModel
)
from qwen_vl_utils import process_vision_info

# ajustes iniciales
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_USE_LEGACY_KERAS"] = "1"
warnings.filterwarnings("ignore")

# mover tensorflow a la cpu para que no sature la grafica
tf.config.set_visible_devices([], "GPU")

# carga de ia para ver imagenes y traducir
model_qwen = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="cuda"
)
processor_qwen = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

translator_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")
translator_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-es").to("cuda")

# carga del clasificador de memes
MODEL_PATH = 'modelo_memes4good_ROBUST.keras'

# MODELOS DISPONIBLES:

# 1. 'modelo_memes4good_BALANCED.keras'   (Versión equilibrada)
# 2. 'modelo_memes4good_ROBUST.keras'     (Versión ligera y robusta)
# 3. 'modelo_memes4good_POWERFUL.keras'   (Modelo + potente)

MAX_LEN = 64 if 'robust' in MODEL_PATH.lower() else 128

tokenizer_bert = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model_juez = tf.keras.models.load_model(
    MODEL_PATH, 
    custom_objects={'TFBertModel': TFBertModel},
    compile=False
)

def _traducir(text):
    # pasar texto de ingles a español
    if not text or not text.strip(): return ""
    inputs = translator_tokenizer(text, return_tensors="pt", padding=True).to("cuda")
    out = translator_model.generate(**inputs, max_new_tokens=200)
    return translator_tokenizer.batch_decode(out, skip_special_tokens=True)[0]

def _get_ocr(image):
    # leer el texto que sale en el meme
    prompt = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Extract all visible text from this image. Return only the transcription."}
    ]}]
    text_in = processor_qwen.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    inputs = processor_qwen(text=[text_in], images=process_vision_info(prompt)[0], return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        generated_ids = model_qwen.generate(**inputs, max_new_tokens=100)
    return processor_qwen.decode(generated_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def _get_description(image):
    # describir que pasa en la imagen
    prompt = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Describe visual context, characters and the irony/joke in one dense sentence. No transcription. No filler. Style: Telegraphic."}
    ]}]
    text_in = processor_qwen.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    inputs = processor_qwen(text=[text_in], images=process_vision_info(prompt)[0], return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        generated_ids = model_qwen.generate(**inputs, max_new_tokens=60, repetition_penalty=1.2)
    out_en = processor_qwen.decode(generated_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return _traducir(out_en)

def _predecir(image, ocr, desc):
    # usar el modelo para decidir si el meme es ofensivo
    img_resized = image.resize((224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img_resized)
    img_batch = np.expand_dims(tf.keras.applications.efficientnet.preprocess_input(img_array), axis=0)

    # recortar texto segun el modelo elegido
    if 'powerful' in MODEL_PATH.lower():
        txt_fusion = f"MEME: {ocr[:200]} [SEP] ESCENA: {desc[:300]}"
    elif 'balanced' in MODEL_PATH.lower():
        txt_fusion = f"MEME: {ocr[:150]} [SEP] ESCENA: {desc[:200]}"
    else:
        txt_fusion = f"MEME: {ocr[:100]} [SEP] ESCENA: {desc[:150]}"

    tokens = tokenizer_bert([txt_fusion], max_length=MAX_LEN, padding="max_length", truncation=True, return_tensors="tf")
    prediction = model_juez.predict({
        "img_input": img_batch,
        "ids_input": tokens["input_ids"],
        "mask_input": tokens["attention_mask"]
    }, verbose=0)[0][0]
    return float(prediction)

2026-02-19 12:46:44.301898: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/esebas/tf4050/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:03<00:00,  1.62s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fa

In [5]:
import gradio as gr

MODEL_NAME = os.path.basename(MODEL_PATH)

def inference_pipeline(input_img):
    if input_img is None: 
        return "<div style='text-align:center; font-size:20px;'>sube una imagen</div>"
    
    ocr = _get_ocr(input_img)
    desc_es = _get_description(input_img)
    score = _predecir(input_img, ocr, desc_es)
    
    es_ofensivo = score < 0.5
    conf = (1 - score) if es_ofensivo else score
    

    alpha = max(0.2, (conf - 0.5) * 2)
    
    if es_ofensivo:
        label = "DAÑINO / OFENSIVO"
        color = f"rgba(255, 0, 0, {alpha})" 
    else:
        label = "INOFENSIVO / SEGURO"
        color = f"rgba(0, 128, 0, {alpha})" 

    return f"""
    <div style="
        text-align: center; 
        padding: 40px; 
        border-radius: 15px; 
        background-color: {color}; 
        color: white; 
        font-family: Arial, sans-serif;
        transition: all 0.5s ease;
    ">
        <p style="font-size: 20px; margin: 0; text-transform: uppercase; opacity: 0.8;">resultado del análisis</p>
        <h1 style="font-size: 50px; margin: 10px 0; font-weight: bold;">{label}</h1>
        <h2 style="font-size: 35px; margin: 0; opacity: 0.9;">Confianza: {conf:.2%}</h2>
    </div>
    """

with gr.Blocks(title="Memes4Good") as demo:
    gr.Markdown(f"<div style='text-align:center;'><h1>Memes4Good - {MODEL_NAME}</h1></div>")
    
    with gr.Column():
        input_image = gr.Image(type="pil", label="sube tu meme")
        submit_btn = gr.Button("analizar meme", variant="primary")
        
        output_html = gr.HTML(label="resultado")

    submit_btn.click(
        fn=inference_pipeline,
        inputs=[input_image],
        outputs=[output_html]
    )

demo.launch(theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
